# Multimodal Duplicate-Listing Detection on Amazon Berkeley Objects

Pipeline: **data → ground truth → baselines (text / image / fused) → contrastive fine-tuning with hard negatives → FAISS at 1M → re-ranker (cross-encoder + SIFT/RANSAC) → failure analysis.**

Run your two setup cells first (GPU check + dataset download). Everything below assumes the tars are extracted in the working directory.

**Important:** ABO has no duplicate labels. Ground truth here is *item-level groups* (all images of one `item_id` = same product) with synthetic title noise. Report only numbers this notebook actually produces.

In [1]:
# Check for GPU
import torch
print("GPU is available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

# Install required dependencies
!pip install -q timm opencv-python-headless scikit-learn matplotlib pandas transformers sentence-transformers faiss-cpu

GPU is available True
Device Name: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.5 MB/s eta 0:00:00


In [2]:
#Download the dataset
!wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar
!wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar
print("Extracting dataset...")
!tar -xf abo-images-small.tar
!tar -xf abo-listings.tar
print("Extraction complete!")

--2026-10-15 11:12:42--  https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar
Resolving amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)... 52.217.170.210, 16.15.246.182, 16.182.72.162, ...
Connecting to amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)|52.217.170.210|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3253381120 (3.0G) [application/x-tar]
Saving to: ‘abo-images-small.tar’

abo-images-small.ta 100%[===================>]   3.03G  12.5MB/s    in 4m 14s  

2026-10-15 11:16:57 (12.2 MB/s) - ‘abo-images-small.tar’ saved [3253381120/3253381120]

--2026-10-15 11:16:57--  https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar
Resolving amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)... 54.231.131.210, 54.231.194.202, 16.15.183.222, ...
Connecting to amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-

In [2]:
import os, gzip, json, glob, random, time, math, functools
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import faiss, cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

CFG = dict(
    seed=42,
    max_items=30000,          # raise for a bigger run (full ABO ~147k items)
    min_imgs=2, max_imgs_per_item=6,
    backbone="openai/clip-vit-base-patch16",   # or "google/siglip-base-patch16-224"
    embed_bs=256, text_len=64,
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## 1. Build listing table + ground truth

In [3]:
def en_values(lst):
    return [d["value"] for d in (lst or []) if d.get("language_tag", "").startswith("en")]

rows = []
for f in sorted(glob.glob("listings/metadata/*.json.gz")):
    with gzip.open(f, "rt") as fh:
        for line in fh:
            d = json.loads(line)
            titles = list(dict.fromkeys(en_values(d.get("item_name"))))
            if not titles: continue
            ids = ([d["main_image_id"]] if d.get("main_image_id") else []) + list(d.get("other_image_id", []))
            if len(ids) < CFG["min_imgs"]: continue
            pt = d.get("product_type") or [{}]
            rows.append(dict(item_id=d["item_id"], titles=titles, product_type=pt[0].get("value"),
                             image_ids=ids[:CFG["max_imgs_per_item"]]))
items = pd.DataFrame(rows).drop_duplicates("item_id")
items = items.sample(min(CFG["max_items"], len(items)), random_state=CFG["seed"]).reset_index(drop=True)
print(len(items), "items")

img_meta = pd.read_csv("images/metadata/images.csv.gz")
path_map = dict(zip(img_meta.image_id, img_meta.path))

rng = random.Random(CFG["seed"])
def perturb(t):
    # synthetic title noise: random word drops + occasional adjacent swap
    w = t.split()
    if len(w) > 4 and rng.random() < 0.5:
        drop = set(rng.sample(range(len(w)), k=max(1, len(w) // 5)))
        w = [x for i, x in enumerate(w) if i not in drop]
    if len(w) > 3 and rng.random() < 0.3:
        i = rng.randrange(len(w) - 1); w[i], w[i + 1] = w[i + 1], w[i]
    return " ".join(w)

L = []
for r in items.itertuples():
    for k, iid in enumerate(r.image_ids):
        p = path_map.get(iid)
        if p is None: continue
        full = f"images/small/{p}"
        if not os.path.exists(full): continue
        L.append(dict(item_id=r.item_id, image_id=iid, path=full,
                      title=perturb(r.titles[k % len(r.titles)]), product_type=r.product_type))
df = pd.DataFrame(L)
df = df[df.groupby("item_id").item_id.transform("size") >= 2].reset_index(drop=True)
df["y"] = pd.factorize(df.item_id)[0]

# split BY ITEM so no product leaks across splits
uitems = df.y.unique(); np.random.shuffle(uitems)
n = len(uitems); split = {}
for i, u in enumerate(uitems):
    split[u] = "train" if i < 0.7 * n else ("val" if i < 0.8 * n else "test")
df["split"] = df.y.map(split)
print(df.split.value_counts(), "\n", len(df), "listings,", df.y.nunique(), "items")

30000 items
split
train    99191
test     28347
val      14317
Name: count, dtype: int64 
 141855 listings, 30000 items


## 2. Frozen CLIP/SigLIP embeddings (cached)

In [4]:
from transformers import AutoModel, AutoProcessor
proc = AutoProcessor.from_pretrained(CFG["backbone"])
backbone = AutoModel.from_pretrained(CFG["backbone"], torch_dtype=torch.float16).to(device).eval()

class ImgDS(torch.utils.data.Dataset):
    def __init__(s, paths): s.p = paths
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return Image.open(s.p[i]).convert("RGB")

def as_tensor(f):
    # older transformers: Tensor. newer: ModelOutput with the embeddings inside
    if torch.is_tensor(f):
        return f
    for name in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(f, name, None)
        if torch.is_tensor(v):
            return v
    raise TypeError(f"Don't know how to get embeddings from {type(f)}; fields: {list(f.keys())}")

@torch.no_grad()
def embed_images(paths):
    dl = torch.utils.data.DataLoader(ImgDS(paths), batch_size=CFG["embed_bs"], num_workers=2,
                                     collate_fn=lambda b: proc(images=b, return_tensors="pt")["pixel_values"])
    out = []
    for px in tqdm(dl, desc="images"):
        f = as_tensor(backbone.get_image_features(pixel_values=px.to(device, torch.float16)))
        out.append(F.normalize(f.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def embed_texts(texts):
    out = []
    for s in tqdm(range(0, len(texts), 512), desc="text"):
        t = proc(text=texts[s:s+512], padding="max_length", max_length=CFG["text_len"],
                 truncation=True, return_tensors="pt").to(device)
        f = as_tensor(backbone.get_text_features(input_ids=t["input_ids"]))
        out.append(F.normalize(f.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

# quick sanity check before the long run: both should print (8, 512) for CLIP B/16
print(embed_images(df.path.tolist()[:8]).shape, embed_texts(df.title.tolist()[:8]).shape)

if os.path.exists("img_emb.npy"):
    img_emb, txt_emb = np.load("img_emb.npy"), np.load("txt_emb.npy")
else:
    img_emb = embed_images(df.path.tolist()); txt_emb = embed_texts(df.title.tolist())
    np.save("img_emb.npy", img_emb); np.save("txt_emb.npy", txt_emb)
df.to_parquet("listings.parquet")
print(img_emb.shape, txt_emb.shape)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

images:   0%|          | 0/1 [00:00<?, ?it/s]

text:   0%|          | 0/1 [00:00<?, ?it/s]

(8, 512) (8, 512)


images:   0%|          | 0/555 [00:00<?, ?it/s]

Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x790bccf5cf40>Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x790bccf5cf40>    
if w.is_alive():Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

AssertionError: can only test a child process
    if w.is_alive():
  File "/usr/lib/

text:   0%|          | 0/278 [00:00<?, ?it/s]

(141855, 512) (141855, 512)


## 3. Evaluation + baselines (text-only / image-only / fused zero-shot)

In [5]:
def eval_recall(emb, y, k=10):
    # recall@k = relevant retrieved / min(k, #relevant); queries with no relevant item are skipped
    emb = np.ascontiguousarray(emb.astype("float32"))
    idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb)
    _, I = idx.search(emb, k + 1)
    cnt = np.bincount(y)[y] - 1
    rec, hit = [], []
    for q in range(len(emb)):
        if cnt[q] == 0: continue
        nn_ = [j for j in I[q] if j != q][:k]
        rel = (y[nn_] == y[q]).sum()
        rec.append(rel / min(k, cnt[q])); hit.append(rel > 0)
    return float(np.mean(rec)), float(np.mean(hit))

def zs_fused(a, b, wi=0.5):
    z = np.concatenate([wi * a, (1 - wi) * b], 1)
    return z / np.linalg.norm(z, axis=1, keepdims=True)

te = np.where(df.split == "test")[0]; va = np.where(df.split == "val")[0]; tr = np.where(df.split == "train")[0]
yte = pd.factorize(df.y.values[te])[0]
results = {}
results["text-only"]  = eval_recall(txt_emb[te], yte)
results["image-only"] = eval_recall(img_emb[te], yte)
results["fused zero-shot"] = eval_recall(zs_fused(img_emb[te], txt_emb[te]), yte)
pd.DataFrame(results, index=["recall@10", "hit@10"]).T

,recall@10,hit@10
text-only,0.792359,0.941616
image-only,0.168341,0.435813
fused zero-shot,0.445182,0.643631


## 4. Fine-tune a fusion head: SupCon loss + hard-negative mining
Backbones stay frozen (fast, fits Colab); a projection + fusion head is trained on cached features. Each epoch re-mines hard negatives: nearest *different* items under the current model are forced into the same batch.

In [6]:
class Fusion(nn.Module):
    def __init__(s, d_in, d=512):
        super().__init__()
        s.ip = nn.Sequential(nn.Linear(d_in, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
        s.tp = nn.Sequential(nn.Linear(d_in, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
        s.out = nn.Linear(2 * d, d)
    def forward(s, xi, xt):
        i = F.normalize(s.ip(xi), dim=-1); t = F.normalize(s.tp(xt), dim=-1)
        return F.normalize(s.out(torch.cat([i, t], -1)) + i + t, dim=-1)

def supcon(z, y, temp=0.07):
    sim = z @ z.T / temp
    eye = torch.eye(len(z), device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(eye, -1e9)
    pos = (y[:, None] == y[None, :]) & ~eye
    logp = sim - torch.logsumexp(sim, 1, keepdim=True)
    npos = pos.sum(1); ok = npos > 0
    return (-(logp * pos).sum(1)[ok] / npos[ok]).mean()

Xi, Xt = torch.tensor(img_emb), torch.tensor(txt_emb)
Y = df.y.values
model = Fusion(img_emb.shape[1]).to(device)

@torch.no_grad()
def embed_all(m):
    m.eval(); out = []
    for s in range(0, len(Xi), 8192):
        out.append(m(Xi[s:s+8192].to(device), Xt[s:s+8192].to(device)).cpu())
    return torch.cat(out).numpy()

item_to_idx = {}
for i in tr: item_to_idx.setdefault(Y[i], []).append(i)
train_items = np.array(list(item_to_idx))

def mine_neighbors(Z, k=8):
    cents = np.stack([Z[item_to_idx[it]].mean(0) for it in train_items]).astype("float32")
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    idx = faiss.IndexFlatIP(cents.shape[1]); idx.add(cents)
    return idx.search(cents, k + 1)[1][:, 1:]

EPOCHS, A = 20, 96
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * math.ceil(len(train_items) / A))
yva = pd.factorize(Y[va])[0]
best, best_state = -1, None
for ep in range(EPOCHS):
    nbr = mine_neighbors(embed_all(model)) if ep >= 1 else None   # epoch 0 = random negatives (warm-up)
    perm = np.random.permutation(len(train_items)); model.train(); tot = 0
    for s in range(0, len(perm), A):
        a = perm[s:s + A]; sel = set(a.tolist())
        if nbr is not None:
            for ai in a: sel.add(int(np.random.choice(nbr[ai, :5])))     # hard negatives
        ids = np.concatenate([np.random.choice(item_to_idx[train_items[i]], 2, replace=False) for i in sel])
        xi, xt = Xi[ids].to(device), Xt[ids].to(device)
        # augmentation: feature noise + modality dropout (teaches robustness to missing/bad title or image)
        xi = xi + 0.02 * torch.randn_like(xi); xt = xt + 0.02 * torch.randn_like(xt)
        r = torch.rand(len(ids), 1, device=device)
        xi = xi * (r > 0.1); xt = xt * ((r < 0.9) | (r > 1.0))
        loss = supcon(model(xi, xt), torch.tensor(Y[ids], device=device))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item()
    Z = embed_all(model); v = eval_recall(Z[va], yva)[0]
    if v > best: best, best_state = v, {k: x.clone() for k, x in model.state_dict().items()}
    print(f"ep {ep:02d} loss {tot / math.ceil(len(perm) / A):.4f} val recall@10 {v:.4f}")
model.load_state_dict(best_state); torch.save(best_state, "fusion.pt")
Z = embed_all(model)
results["fused fine-tuned"] = eval_recall(Z[te], yte)
pd.DataFrame(results, index=["recall@10", "hit@10"]).T

ep 00 loss 1.7539 val recall@10 0.8917
ep 01 loss 1.6985 val recall@10 0.9223
ep 02 loss 1.5754 val recall@10 0.9320
ep 03 loss 1.5484 val recall@10 0.9348
ep 04 loss 1.5421 val recall@10 0.9372
ep 05 loss 1.5462 val recall@10 0.9391
ep 06 loss 1.4990 val recall@10 0.9405
ep 07 loss 1.4903 val recall@10 0.9421
ep 08 loss 1.4625 val recall@10 0.9428
ep 09 loss 1.4515 val recall@10 0.9450
ep 10 loss 1.4278 val recall@10 0.9470
ep 11 loss 1.3996 val recall@10 0.9484
ep 12 loss 1.3976 val recall@10 0.9497
ep 13 loss 1.3560 val recall@10 0.9512
ep 14 loss 1.3546 val recall@10 0.9511
ep 15 loss 1.3193 val recall@10 0.9523
ep 16 loss 1.2967 val recall@10 0.9529
ep 17 loss 1.2844 val recall@10 0.9532
ep 18 loss 1.2728 val recall@10 0.9535
ep 19 loss 1.2786 val recall@10 0.9535


,recall@10,hit@10
text-only,0.792359,0.941616
image-only,0.168341,0.435813
fused zero-shot,0.445182,0.643631
fused fine-tuned,0.929519,0.987159


## 5. FAISS at 1M scale (real test embeddings + synthetic distractors)
Distractors are convex mixes of two random real embeddings plus noise: a cheap stand-in for a 1M-listing catalogue. They are labelled non-duplicates, so they only make retrieval harder.

In [7]:
def make_distractors(real, n, sigma=0.3, seed=0):
    g = torch.Generator().manual_seed(seed); R = torch.tensor(real); d = R.shape[1]; out = []
    for s in range(0, n, 100_000):
        m = min(100_000, n - s)
        a = R[torch.randint(len(R), (m,), generator=g)]; b = R[torch.randint(len(R), (m,), generator=g)]
        lam = torch.rand(m, 1, generator=g) * 0.4 + 0.3
        z = lam * a + (1 - lam) * b + sigma / math.sqrt(d) * torch.randn(m, d, generator=g)
        out.append(F.normalize(z, dim=-1))
    return torch.cat(out).numpy()

N_TOTAL = 1_000_000
Zte = np.ascontiguousarray(Z[te].astype("float32"))
dis = make_distractors(Z, N_TOTAL - len(Zte))
d = Zte.shape[1]

quant = faiss.IndexFlatIP(d)
index = faiss.IndexIVFPQ(quant, d, 4096, 64, 8, faiss.METRIC_INNER_PRODUCT)
sample = np.concatenate([Zte, dis[np.random.choice(len(dis), 200_000, replace=False)]])
t = time.time(); index.train(sample); index.add(Zte); index.add(dis)
print(f"built {index.ntotal:,} vectors in {time.time() - t:.0f}s")

def eval_big(index, nprobe, nq=2000, k=10):
    index.nprobe = nprobe
    qs = np.random.RandomState(0).choice(len(Zte), min(nq, len(Zte)), replace=False)
    cnt = np.bincount(yte)[yte] - 1; lat, rec = [], []
    for q in qs:
        t0 = time.perf_counter(); _, I = index.search(Zte[q:q+1], k + 1); lat.append((time.perf_counter() - t0) * 1e3)
        if cnt[q] == 0: continue
        nn_ = [j for j in I[0] if j != q and j >= 0][:k]
        rel = sum(1 for j in nn_ if j < len(Zte) and yte[j] == yte[q])
        rec.append(rel / min(k, cnt[q]))
    return dict(nprobe=nprobe, recall_at_10=np.mean(rec), p50_ms=np.percentile(lat, 50), p95_ms=np.percentile(lat, 95))

pd.DataFrame([eval_big(index, p) for p in [4, 8, 16, 32, 64]])

built 1,000,000 vectors in 152s


,nprobe,recall_at_10,p50_ms,p95_ms
0,4,0.834625,0.445613,0.601031
1,8,0.845333,0.510665,0.625388
2,16,0.846833,0.622084,0.799871
3,32,0.847533,0.857337,1.085844
4,64,0.847533,1.281939,1.612611


## 6. Stage-2 re-ranker: title cross-encoder + SIFT/RANSAC geometric verification

In [8]:
# ---- 6a. Cross-encoder on title pairs (hard negatives from the fine-tuned embedding space) ----
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CE_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ce_tok = AutoTokenizer.from_pretrained(CE_NAME)
ce = AutoModelForSequenceClassification.from_pretrained(CE_NAME, num_labels=1).to(device)

titles = df.title.values
Ztr = np.ascontiguousarray(Z[tr].astype("float32"))
tidx = faiss.IndexFlatIP(Ztr.shape[1]); tidx.add(Ztr); _, NI = tidx.search(Ztr, 11)
pairs = []
for qi, row in enumerate(NI):
    g = tr[qi]
    for nj in row[1:]:
        h = tr[nj]
        pairs.append((titles[g], titles[h], float(Y[g] == Y[h])))
pairs = [p for p in pairs if True]
random.shuffle(pairs); pairs = pairs[:120_000]
print(len(pairs), "pairs, positive rate", np.mean([p[2] for p in pairs]))

opt_ce = torch.optim.AdamW(ce.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")
ce.train()
for s in tqdm(range(0, len(pairs), 64), desc="cross-encoder"):
    b = pairs[s:s+64]
    enc = ce_tok([x[0] for x in b], [x[1] for x in b], padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
    with torch.autocast(device, enabled=device == "cuda"):
        logit = ce(**enc).logits.squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logit.float(), torch.tensor([x[2] for x in b], device=device))
    opt_ce.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt_ce); scaler.update()
ce.eval()

@torch.no_grad()
def ce_score(ta, tb, bs=256):
    out = []
    for s in range(0, len(ta), bs):
        enc = ce_tok(ta[s:s+bs], tb[s:s+bs], padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
        out.append(ce(**enc).logits.squeeze(-1).float().cpu())
    return torch.cat(out).numpy()

# ---- 6b. Local-feature geometric verification (SIFT + ratio test + RANSAC homography) ----
sift = cv2.SIFT_create(nfeatures=300); bf = cv2.BFMatcher()

@functools.lru_cache(maxsize=3000)
def feats(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    kp, des = sift.detectAndCompute(g, None)
    return np.float32([k.pt for k in kp]) if kp else np.zeros((0, 2), np.float32), des

def geo_inliers(pa, pb):
    ka, da = feats(pa); kb, db = feats(pb)
    if da is None or db is None or len(da) < 8 or len(db) < 8: return 0
    good = [m[0] for m in bf.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < 0.75 * m[1].distance]
    if len(good) < 8: return 0
    src = ka[[g.queryIdx for g in good]].reshape(-1, 1, 2); dst = kb[[g.trainIdx for g in good]].reshape(-1, 1, 2)
    _, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return int(mask.sum()) if mask is not None else 0

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

120000 pairs, positive rate 0.34269166666666667


/tmp/ipykernel_3531/2242867473.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")


cross-encoder:   0%|          | 0/1875 [00:00<?, ?it/s]

### 6c. Candidate pairs, feature fusion, precision @ 90% recall

In [9]:
from sklearn.linear_model import LogisticRegression

def build_candidates(split_idx, nq=1000, K=20, seed=0):
    Zs = np.ascontiguousarray(Z[split_idx].astype("float32")); ys = Y[split_idx]
    ix = faiss.IndexFlatIP(Zs.shape[1]); ix.add(Zs)
    qs = np.random.RandomState(seed).choice(len(Zs), min(nq, len(Zs)), replace=False)
    S, I = ix.search(Zs[qs], K + 1)
    recs = []
    for r, q in enumerate(qs):
        for s, j in zip(S[r], I[r]):
            if j == q: continue
            recs.append((split_idx[q], split_idx[j], float(s), float(ys[q] == ys[j])))
    c = pd.DataFrame(recs, columns=["q", "c", "sim", "label"])
    total_pos = int(sum((np.bincount(ys)[ys[q]] - 1) for q in qs))
    c["ce"] = ce_score(titles[c.q].tolist(), titles[c.c].tolist())
    paths = df.path.values
    c["inl"] = [geo_inliers(paths[a], paths[b]) for a, b in tqdm(zip(c.q, c.c), total=len(c), desc="geom verify")]
    return c, total_pos

cand_va, pos_va = build_candidates(va)
cand_te, pos_te = build_candidates(te, seed=1)

def feat(c): return np.c_[c.sim, c.ce, np.log1p(c.inl), (c.inl >= 15).astype(float)]
def prec_at_recall(score, label, total_pos, target=0.9):
    o = np.argsort(-score); tp = np.cumsum(label[o]); rec = tp / total_pos
    k = np.searchsorted(rec, target)
    return float(tp[k] / (k + 1)) if k < len(o) else float("nan")

print(f"candidate recall ceiling on test (top-20): {cand_te.label.sum() / pos_te:.3f}")
out = {"embedding sim only": prec_at_recall(cand_te.sim.values, cand_te.label.values, pos_te)}
for name, cols in {"+ cross-encoder": [0, 1], "+ geometry": [0, 2, 3], "+ both": [0, 1, 2, 3]}.items():
    lr = LogisticRegression(max_iter=1000).fit(feat(cand_va)[:, cols], cand_va.label)
    out[name] = prec_at_recall(lr.decision_function(feat(cand_te)[:, cols]), cand_te.label.values, pos_te)
pd.Series(out, name="precision @ 90% recall")

geom verify:   0%|          | 0/20000 [00:00<?, ?it/s]

geom verify:   0%|          | 0/20000 [00:00<?, ?it/s]

candidate recall ceiling on test (top-20): 0.969


,precision @ 90% recall
embedding sim only,0.472848
+ cross-encoder,0.909091
+ geometry,0.476063
+ both,0.906782


## 7. Failure analysis: viewpoint, background, lighting
Proxies computed from the image pairs: SIFT inlier count (low = large viewpoint change), border-whiteness difference (background), mean-brightness difference (lighting).

In [ ]:
@functools.lru_cache(maxsize=None)
def img_stats(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE); h, w = g.shape; b = max(2, h // 10)
    border = np.concatenate([g[:b].ravel(), g[-b:].ravel(), g[:, :b].ravel(), g[:, -b:].ravel()])
    return g.mean(), (border > 240).mean()

Zs = np.ascontiguousarray(Z[te].astype("float32")); ix = faiss.IndexFlatIP(Zs.shape[1]); ix.add(Zs)
qs = np.random.RandomState(2).choice(len(Zs), min(1500, len(Zs)), replace=False)
_, I = ix.search(Zs[qs], 11)
rows = []
for r, q in enumerate(qs):
    top = set(I[r].tolist()) - {q}
    for p in np.where(yte == yte[q])[0]:
        if p == q: continue
        a, b = df.path.values[te[q]], df.path.values[te[p]]
        (ba, wa), (bb, wb) = img_stats(a), img_stats(b)
        rows.append(dict(retrieved=p in top, d_light=abs(ba - bb), d_bg=abs(wa - wb), inl=geo_inliers(a, b)))
fa = pd.DataFrame(rows)
fa["viewpoint"] = pd.cut(fa.inl, [-1, 0, 9, 24, 1e9], labels=["0", "1-9", "10-24", "25+"])
fa["lighting"] = pd.qcut(fa.d_light, 4, duplicates="drop")
fa["background"] = pd.qcut(fa.d_bg.rank(method="first"), 4, labels=["same", "slight", "different", "very different"])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for a, c, t in zip(ax, ["viewpoint", "lighting", "background"],
                   ["Shared local features (viewpoint proxy)", "Brightness gap", "Background gap"]):
    fa.groupby(c, observed=True).retrieved.mean().plot.bar(ax=a, title=t); a.set_xlabel("")
ax[0].set_ylabel("P(true duplicate in top-10)"); plt.tight_layout(); plt.show()
print(len(fa), "true-positive pairs analysed; overall retrieved rate", fa.retrieved.mean().round(3))

## 8. What to do next
- Write only the numbers from sections 3, 5 and 6c into your resume bullets.
- Scale: raise `max_items`, try `google/siglip-base-patch16-224`, unfreeze the last backbone blocks (LoRA) for end-to-end image augmentation.
- Pull the worst failures with `fa[~fa.retrieved]` and view the image pairs for the write-up.